
# Episode 3 — Sport Agent Showdown

**Question:** can one sports agent answer football, F1 and chess questions without mixing facts?

This notebook tells the end-to-end story behind Episode 3. We compare four agentic designs:

1. `single_agent`
2. `sequential_chain`
3. `triage_handoff`
4. `committee_referee`

The task is intentionally multi-domain: World Cup 2026 groups, Formula 1 2026 schedules/lineups, chess ratings, and legacy football player stats.



## 1. Setup

The notebook runs in deterministic offline mode. The optional live OpenAI path is implemented in `src/sport_agent/openai_agents.py`, but not used here so the experiment is reproducible without API cost.


In [ ]:

from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown, Image

ROOT = Path.cwd()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from sport_agent.data import world_cup_groups, f1_calendar, f1_lineups, chess_top_players, source_cards
from sport_agent.designs import run_design, DESIGNS
from sport_agent.eval_harness import evaluate_design, summarise, load_eval_cases

print(ROOT)



## 2. Data snapshot

This is not a live sports database. It is a compact, source-backed snapshot used for a controlled agent-engineering experiment.


In [ ]:

wc = world_cup_groups()['groups']
teams = [team for group in wc.values() for team in group]
summary = {
    'World Cup groups': len(wc),
    'World Cup teams': len(teams),
    'Italy in World Cup snapshot?': 'Italy' in teams,
    'F1 current schedule rounds': len(f1_calendar()['rounds']),
    'F1 teams in line-up snapshot': len(f1_lineups()['lineups']),
    'Chess players in rating snapshot': len(chess_top_players()['players']),
    'RAG source cards': len(source_cards()),
}
display(pd.DataFrame([summary]).T.rename(columns={0:'value'}))



## 3. Example questions

The benchmark contains questions that force routing, retrieval, source conflicts, abstention, and mixed-sport decomposition.


In [ ]:

cases = load_eval_cases()
display(pd.DataFrame([{
    'case_id': c.case_id,
    'sports': ', '.join(c.expected_sports),
    'question': c.question,
    'should_abstain': c.should_abstain,
} for c in cases]).head(15))



## 4. Same question, four designs

This mixed briefing question is where a single generalist starts to break down. The handoff and committee designs explicitly split the work by sport.


In [ ]:

question = 'Build me a mini sports briefing: World Cup Group D, the next F1 race, and the chess number one.'
rows = []
for design in DESIGNS:
    ans = run_design(question, design)
    rows.append({
        'design': design,
        'sports': ', '.join(ans.sports),
        'tools': ', '.join(ans.tools_used),
        'skills': ', '.join(ans.skills_used),
        'citations': ', '.join(ans.citations),
        'abstained': ans.abstained,
        'answer': ans.answer,
    })
display(pd.DataFrame(rows))



## 5. Run the evaluation harness

The harness checks route accuracy, tool recall, skill recall, citation recall, expected keywords, forbidden claims, abstention accuracy, unsupported claims, and cost/tool-call proxies.


In [ ]:

summaries = []
all_results = {}
for design in DESIGNS:
    results = evaluate_design(design)
    all_results[design] = results
    summaries.append(summarise(results))
summary_df = pd.DataFrame(summaries)
display(summary_df)



## 6. Visual comparison

The figures below are generated by `scripts/make_figures.py` after evaluation. They show that `triage_handoff` gets the best quality/cost balance in this deterministic benchmark, while `committee_referee` matches quality with more tool calls.


In [ ]:

for img in [
    'reports/figures/design_pass_rate.png',
    'reports/figures/tool_cost_tradeoff.png',
    'reports/figures/metric_radar_like.png',
]:
    display(Markdown(f'### {Path(img).name}'))
    display(Image(filename=str(ROOT / img)))



## 7. Failure analysis

The single-agent design looks simple, but the eval makes its weaknesses visible: weak tool discipline, incomplete citations, and poor handling of mixed-domain questions.


In [ ]:

failure_rows = []
for design, results in all_results.items():
    for r in results:
        if not r.passed:
            failure_rows.append({
                'design': design,
                'case_id': r.case_id,
                'failures': ', '.join(r.failures),
                'answer': r.answer.answer,
            })
display(pd.DataFrame(failure_rows).head(20))



## 8. Design recommendation

For this sport-agent scope:

- **Use `triage_handoff` as the practical default.** It gets perfect eval performance here at lower cost than committee-referee.
- **Use `committee_referee` for higher-stakes answers.** It adds verification and is easier to extend with stricter checks.
- **Avoid relying on a single generalist** for broad sports questions with current facts and mixed domains.

This is the episode's main engineering point: the interesting part is not just adding more data. It is choosing the right agent orchestration pattern for the failure modes.
